# Missouri recorded-wildfire analysis: documented workflow

This notebook was reorganized on 2026-10-04 without rerunning its analyses.
Retained outputs and execution counts are historical, from the original notebook;
they do not certify execution of the edited source. Edited cells record their
original identifiers and source hashes in `metadata.cleanup_provenance`.
The exact original notebook, including superseded calculations and outputs, is
preserved at `archive/pre_publication_cleanup_2026-10-03/etl.original.ipynb`.
`docs/notebook_cell_changes.csv` maps every original cell to this version.

The response is **at least one recorded wildfire during 2021–2025** in a 5-km
modeling cell. Model scores and their quintiles describe relative susceptibility
to that recorded response; they are not established calibrated probabilities of
future wildfire. Zero denotes no recorded wildfire, not confirmed absence.

The analyses below remain distinct: baseline spatial `GroupKFold` (reported AUC
0.673), stratified random CV (0.768), stratified spatial baseline (0.657), globally
tuned nonnested spatial CV, and nested RF spatial CV. The full-data RF used for
SHAP and mapping retains the configuration selected by the earlier spatial grid
search. It is not replaced by an outer-fold model from nested CV.

**Execution scope.** This is the complete historical analysis workflow; Run All
would refit models and overwrite historical output paths. Publication generation
is separated into `scripts/make_publication_figures.py`, and selective validation
into `scripts/reproduce_validation.py`. Use their documented explicit commands
when needed; opening this notebook does not execute them. No full-notebook rerun
was performed during cleanup.

## 1. Inputs and original quality-control evidence

The following inspection outputs document the source fire records, geometry,
dates, duplicate identifiers, and study boundary. They are retained as provenance.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask


In [ ]:
file_path = "data/InFORM_FireOccurrence_MO.gpkg"

fires = gpd.read_file(file_path)

print(fires.shape)
fires.head()

In [ ]:
# columns
fires.columns.tolist()

In [ ]:
# columns info
fires.info()

count fires by year

In [ ]:
fires["CalendarYear"].describe()

In [ ]:
year_counts = (
    fires.groupby("CalendarYear")
    .size()
    .reset_index(name="Fire_Count")
)

print(year_counts)

In [ ]:
plt.figure(figsize=(12, 5))

plt.bar(
    year_counts["CalendarYear"],
    year_counts["Fire_Count"]
)

plt.xlabel("Year")
plt.ylabel("Number of Wildfire Records")
plt.title("Missouri Wildfire Records by Year")

plt.show()

In [ ]:
fires[
    fires["InitialLatitude"].isna() |
    fires["InitialLongitude"].isna()
].shape

In [ ]:
fires[
    fires["InitialLatitude"].isna() |
    fires["InitialLongitude"].isna()
][
    ["CalendarYear", "IncidentName",
     "InitialLatitude", "InitialLongitude"]
].head(20)

In [ ]:
fires.total_bounds

check duplicated

In [ ]:
fires["UniqueFireIdentifier"].duplicated().sum()

In [ ]:
duplicates = fires[
    fires["UniqueFireIdentifier"].duplicated(keep=False)
]

duplicates[
    ["UniqueFireIdentifier",
     "CalendarYear",
     "IncidentName"]
].sort_values("UniqueFireIdentifier").head(30)

examine fire size

In [ ]:
fires["IncidentSize"].describe()

In [ ]:
fires["IncidentSize"].isna().sum()

In [ ]:
import numpy as np

size = fires["IncidentSize"].dropna()
size = size[size > 0]

plt.figure(figsize=(9, 5))

plt.hist(np.log10(size), bins=40)

plt.xlabel("log10(Fire Size in Acres)")
plt.ylabel("Number of Fires")
plt.title("Distribution of Missouri Wildfire Size")

plt.show()

map the wildfire points

In [ ]:
fires.plot(
    figsize=(8, 8),
    markersize=2
)

plt.title("Historical Wildfire Occurrences in Missouri")
plt.show()

In [ ]:
boundary = gpd.read_file(
    "data/MO_State_Boundary/MO_State_Boundary.shp"
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

boundary.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black"
)

fires.plot(
    ax=ax,
    markersize=2
)

plt.title("Missouri Historical Wildfire Occurrences")
plt.show()

In [ ]:
fires.shape

In [ ]:
fires.columns.to_list()

In [ ]:
fires['CalendarYear'].describe()

In [ ]:
year_counts

Quality check

In [ ]:
# Missing coordinates
missing_coords = fires[
    fires["InitialLatitude"].isna() |
    fires["InitialLongitude"].isna()
]

print("Missing coordinates:", len(missing_coords))

In [ ]:
# Duplicate fire identifiers
print(
    "Duplicate IDs:",
    fires["UniqueFireIdentifier"].duplicated().sum()
)

In [ ]:
# Fire size
fires["IncidentSize"].describe()

In [ ]:
pd.crosstab(
    fires["CalendarYear"],
    fires["CreatedBySystem"]
)

investigate those 3,714 duplicates

In [ ]:
dup = fires[
    fires["UniqueFireIdentifier"].duplicated(keep=False)
].sort_values("UniqueFireIdentifier")

print("Rows involved in duplicate IDs:", len(dup))
print("Unique duplicated IDs:",
      dup["UniqueFireIdentifier"].nunique())

In [ ]:
cols = [
    "UniqueFireIdentifier",
    "CalendarYear",
    "IncidentName",
    "FireDiscoveryDateTime",
    "InitialLatitude",
    "InitialLongitude",
    "IncidentSize",
    "CreatedBySystem"
]

dup[cols].head(30)

identify those 6 values

In [ ]:
dup_counts = (
    fires["UniqueFireIdentifier"]
    .value_counts(dropna=False)
)

dup_counts[dup_counts > 1]

In [ ]:
duplicated_ids = dup_counts[dup_counts > 1].index

fires[
    fires["UniqueFireIdentifier"].isin(duplicated_ids)
][[
    "UniqueFireIdentifier",
    "CalendarYear",
    "IncidentName",
    "FireDiscoveryDateTime",
    "InitialLatitude",
    "InitialLongitude",
    "IncidentSize",
    "CreatedBySystem"
]].head(50)

In [ ]:
fires["UniqueFireIdentifier"].isna().sum()

inspect those 10 rows

In [ ]:
real_dup_ids = [
    "2016-MOMTF-000207",
    "2017-MOMTF-000182",
    "2017-MOMTF-000434",
    "2024-MOMOS-000037",
    "2025-MOMTF-000429"
]

fires[
    fires["UniqueFireIdentifier"].isin(real_dup_ids)
][[
    "UniqueFireIdentifier",
    "CalendarYear",
    "IncidentName",
    "FireDiscoveryDateTime",
    "InitialLatitude",
    "InitialLongitude",
    "IncidentSize",
    "CreatedBySystem"
]].sort_values("UniqueFireIdentifier")

Spatial validation

In [ ]:
boundary = gpd.read_file(
    "data/MO_State_Boundary/MO_State_Boundary.shp"
)

boundary

In [ ]:
print("Boundary CRS:", boundary.crs)
print("Fire CRS:", fires.crs)

In [ ]:
# reomove recoreds without usable geometry
fires_geo = fires[
    fires.geometry.notna() &
    ~fires.geometry.is_empty
].copy()

print("Original records:", len(fires))
print("Records with geometry:", len(fires_geo))
print("Removed:", len(fires) - len(fires_geo))

In [ ]:
missouri = boundary.geometry.union_all()

fires_geo["inside_missouri"] = fires_geo.geometry.intersects(missouri)

fires_geo["inside_missouri"].value_counts()

In [ ]:
outside = fires_geo[~fires_geo["inside_missouri"]]

print("Outside Missouri:", len(outside))

In [ ]:
outside[[
    "CalendarYear",
    "IncidentName",
    "InitialLatitude",
    "InitialLongitude",
    "IncidentSize",
    "CreatedBySystem"
]].head(30)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

boundary.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=1
)

fires_geo.plot(
    ax=ax,
    markersize=2,
    alpha=0.4
)

ax.set_title("Historical Wildfire Occurrences in Missouri")
ax.set_axis_off()

plt.show()

verify the missing geometries 66 records

In [ ]:
missing_coords = fires[
    fires["InitialLatitude"].isna() |
    fires["InitialLongitude"].isna()
]

print("Missing InitialLat/Lon:", len(missing_coords))

missing_coords[[
    "IncidentName",
    "CalendarYear",
    "InitialLatitude",
    "InitialLongitude",
    "geometry"
]].head(10)

In [ ]:
print("Missing geometry:",
      missing_coords.geometry.isna().sum())

print("Empty geometry:",
      missing_coords.geometry.is_empty.sum())

print("Valid geometry:",
      missing_coords.geometry.is_valid.sum())

In [ ]:
fires.geometry

In [ ]:
fires["InitialLatitude"]
fires["InitialLongitude"]

visualize the wildfire distribution

In [ ]:
# Plot Missouri boundary and historical wildfire locations
fig, ax = plt.subplots(figsize=(9, 9))

# Draw the Missouri state boundary
boundary.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=1.2
)

# Add wildfire occurrence points
fires.plot(
    ax=ax,
    markersize=2,
    alpha=0.35
)

# Add title and remove map axes
ax.set_title("Historical Wildfire Occurrences in Missouri")
ax.set_axis_off()

plt.show()

## 2. Modeling grid and predictor aggregation

Retain the original 5-km grid, fire assignment, Missouri intersection, and final
aggregation methods. The superseded un-clipped continuous aggregation and first
population attempt are available in the archived original notebook. The retained
population calculation treats masked raster pixels as zero exactly as before.

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask

# ----------------------------------------
# Clip LANDFIRE FBFM40 to Missouri
# ----------------------------------------

# Path to the downloaded LANDFIRE raster
fbfm_path = r"data/LF2024_FBFM40_CONUS/LF2024_FBFM40_CONUS.tif"

# Output Missouri-only raster
output_path = r"Missouri_FBFM40_2024.tif"

# Convert the existing Missouri Polygon
# back into a GeoDataFrame
missouri_gdf = gpd.GeoDataFrame(
    geometry=[missouri],
    crs="EPSG:3857"
)

# Open the CONUS FBFM40 raster
with rasterio.open(fbfm_path) as src:

    print("FBFM40 CRS:", src.crs)
    print("Raster size:", src.width, "x", src.height)

    # Reproject Missouri to match LANDFIRE
    missouri_fbfm = missouri_gdf.to_crs(src.crs)

    # Clip the raster to Missouri
    clipped, transform = mask(
        src,
        missouri_fbfm.geometry,
        crop=True
    )

    # Copy original raster metadata
    metadata = src.meta.copy()

    # Update metadata for clipped raster
    metadata.update({
        "height": clipped.shape[1],
        "width": clipped.shape[2],
        "transform": transform
    })

# Save the Missouri-only raster
with rasterio.open(output_path, "w", **metadata) as dst:
    dst.write(clipped)

print("Saved:", output_path)

In [ ]:
import rasterio
import numpy as np

# Open the clipped Missouri FBFM40 raster
with rasterio.open("Missouri_FBFM40_2024.tif") as src:
    data = src.read(1)

    print("CRS:", src.crs)
    print("Size:", src.width, "x", src.height)
    print("Resolution:", src.res)
    print("NoData value:", src.nodata)

    # Get unique fuel-model values, excluding NoData
    if src.nodata is not None:
        values = np.unique(data[data != src.nodata])
    else:
        values = np.unique(data)

    print("Number of unique values:", len(values))
    print("Unique values:", values[:50])

In [ ]:
import rasterio

# Compare CONUS and Missouri raster bounds
with rasterio.open(fbfm_path) as src:
    print("CONUS bounds:", src.bounds)

with rasterio.open("Missouri_FBFM40_2024.tif") as src:
    print("Missouri bounds:", src.bounds)
    print("Missouri size:", src.width, "x", src.height)

In [ ]:
import rasterio
import numpy as np

# Check how much of the raster is NoData
with rasterio.open("Missouri_FBFM40_2024.tif") as src:
    data = src.read(1)

    nodata = src.nodata

    # Count valid and NoData pixels
    valid_pixels = np.sum(data != nodata)
    nodata_pixels = np.sum(data == nodata)
    total_pixels = data.size

    print("Total pixels:", total_pixels)
    print("Valid pixels:", valid_pixels)
    print("NoData pixels:", nodata_pixels)
    print(
        "NoData percentage:",
        round(nodata_pixels / total_pixels * 100, 2),
        "%"
    )

In [ ]:
import geopandas as gpd
from shapely.geometry import box
import numpy as np

# ----------------------------------------
# Create 5 km Missouri modeling grid
# ----------------------------------------

# Reproject Missouri boundary to EPSG:5070
missouri_5070 = missouri_gdf.to_crs("EPSG:5070")

# Grid cell size: 5 km = 5,000 meters
cell_size = 5000

# Get Missouri bounding box
minx, miny, maxx, maxy = missouri_5070.total_bounds

# Create grid coordinates
x_coords = np.arange(minx, maxx, cell_size)
y_coords = np.arange(miny, maxy, cell_size)

# Create 5 km square cells
grid_cells = []

for x in x_coords:
    for y in y_coords:
        grid_cells.append(
            box(
                x,
                y,
                x + cell_size,
                y + cell_size
            )
        )

# Convert cells to GeoDataFrame
grid = gpd.GeoDataFrame(
    geometry=grid_cells,
    crs="EPSG:5070"
)

# Keep cells whose centroid falls inside Missouri
grid["centroid"] = grid.geometry.centroid

grid = grid[
    grid["centroid"].within(
        missouri_5070.geometry.iloc[0]
    )
].copy()

# Remove temporary centroid column
grid = grid.drop(columns="centroid")

# Add unique grid ID
grid["grid_id"] = range(1, len(grid) + 1)

print("Number of 5 km cells:", len(grid))
print("CRS:", grid.crs)

In [ ]:
# ----------------------------------------
# Prepare 2021-2025 wildfire points
# ----------------------------------------

# Keep the selected study period
fires_model = fires_geo[
    fires_geo["CalendarYear"].between(2021, 2025)
].copy()

# Reproject wildfire points to match the grid
fires_model = fires_model.to_crs("EPSG:5070")

print("Wildfires 2021-2025:", len(fires_model))
print("Fire CRS:", fires_model.crs)

In [ ]:
# ----------------------------------------
# Count wildfires in each 5 km grid cell
# ----------------------------------------

# Spatially join wildfire points to grid cells
fire_join = gpd.sjoin(
    fires_model,
    grid[["grid_id", "geometry"]],
    how="inner",
    predicate="within"
)

# Count fires in each grid cell
fire_counts = (
    fire_join.groupby("grid_id")
    .size()
    .rename("fire_count")
)

# Add counts back to the full grid
grid = grid.join(
    fire_counts,
    on="grid_id"
)

# Cells with no recorded fires get a count of zero
grid["fire_count"] = grid["fire_count"].fillna(0).astype(int)

print("Total fires assigned to grid:", grid["fire_count"].sum())
print("Cells with >= 1 fire:", (grid["fire_count"] > 0).sum())
print("Cells with 0 fires:", (grid["fire_count"] == 0).sum())
print("Maximum fires in one cell:", grid["fire_count"].max())

In [ ]:
import rasterio

# ----------------------------------------
# Inspect exported predictor rasters
# ----------------------------------------

files = {
    "continuous": "data/Missouri_Continuous_Predictors_250m.tif",
    "landcover": "data/Missouri_NLCD_2021_250m.tif",
    "population": "data/Missouri_WorldPop_2020_100m.tif",
}

for name, path in files.items():

    with rasterio.open(path) as src:

        print(f"\n{name.upper()}")
        print("CRS:", src.crs)
        print("Size:", src.width, "x", src.height)
        print("Resolution:", src.res)
        print("Bands:", src.count)
        print("Band names:", src.descriptions)
        print("NoData:", src.nodata)
        print("Bounds:", src.bounds)

In [ ]:
# ----------------------------------------
# Create Missouri-only area inside each grid cell
# ----------------------------------------

# Keep original 5 km square geometry
grid["grid_square"] = grid.geometry.copy()

# Intersect each grid cell with Missouri
grid["geometry"] = grid.geometry.intersection(
    missouri_5070.geometry.iloc[0]
)

# Remove any empty geometries
grid = grid[~grid.geometry.is_empty].copy()

print("Grid cells:", len(grid))
print("CRS:", grid.crs)

In [ ]:
import rasterio
import numpy as np
import pandas as pd
from rasterio.mask import mask

# ----------------------------------------
# Aggregate continuous predictors to grid
# ----------------------------------------

continuous_path = "data/Missouri_Continuous_Predictors_250m.tif"

with rasterio.open(continuous_path) as src:

    band_names = list(src.descriptions)
    predictor_rows = []

    for _, row in grid.iterrows():

        # Extract pixels inside Missouri portion of cell
        out_image, _ = mask(
            src,
            [row.geometry],
            crop=True,
            filled=False
        )

        values = []

        for band in out_image:

            # Keep valid pixels
            valid = band.compressed()

            if len(valid) > 0:
                values.append(valid.mean())
            else:
                values.append(np.nan)

        predictor_rows.append(
            [row["grid_id"]] + values
        )

# Create predictor table
continuous_df = pd.DataFrame(
    predictor_rows,
    columns=["grid_id"] + band_names
)

print("Rows:", len(continuous_df))

print("\nMissing values:")
print(continuous_df.isna().sum())

In [ ]:
# ----------------------------------------
# Inspect the 5 remaining missing cells
# ----------------------------------------

missing_ids = continuous_df.loc[
    continuous_df["elevation"].isna(),
    "grid_id"
]

missing_check = grid[
    grid["grid_id"].isin(missing_ids)
][["grid_id", "fire_count", "geometry"]].copy()

# Calculate actual area inside Missouri
missing_check["area_km2"] = (
    missing_check.geometry.area / 1_000_000
)

print(
    missing_check[
        ["grid_id", "fire_count", "area_km2"]
    ]
)

print("\nTotal fires:", missing_check["fire_count"].sum())

In [ ]:
import rasterio
import numpy as np
import pandas as pd
from rasterio.mask import mask

# ----------------------------------------
# Dominant NLCD class in each 5 km cell
# ----------------------------------------

landcover_path = "data/Missouri_NLCD_2021_250m.tif"

landcover_rows = []

with rasterio.open(landcover_path) as src:

    for _, row in grid.iterrows():

        # Extract pixels inside the Missouri portion
        out_image, _ = mask(
            src,
            [row.geometry],
            crop=True,
            filled=False
        )

        # Get valid land-cover pixels
        values = out_image[0].compressed()

        if len(values) > 0:

            # Find the most common NLCD class
            classes, counts = np.unique(
                values.astype(int),
                return_counts=True
            )

            dominant_class = classes[
                np.argmax(counts)
            ]

        else:
            dominant_class = np.nan

        landcover_rows.append([
            row["grid_id"],
            dominant_class
        ])

# Create land-cover table
landcover_df = pd.DataFrame(
    landcover_rows,
    columns=["grid_id", "landcover"]
)

print("Rows:", len(landcover_df))
print("Missing:", landcover_df["landcover"].isna().sum())

print("\nDominant land-cover classes:")
print(
    landcover_df["landcover"]
    .value_counts()
    .sort_index()
)

In [ ]:
# ----------------------------------------
# Aggregate population to 5 km grid
# Treat masked WorldPop pixels as zero
# ----------------------------------------

population_path = "data/Missouri_WorldPop_2020_100m.tif"

population_rows = []

with rasterio.open(population_path) as src:

    for _, row in grid.iterrows():

        # Extract population pixels for this cell
        out_image, _ = mask(
            src,
            [row.geometry],
            crop=True,
            filled=True,
            nodata=0
        )

        # Population band
        values = out_image[0]

        # Sum population in the cell
        population = np.nansum(values)

        population_rows.append([
            row["grid_id"],
            population
        ])

# Create population table
population_df = pd.DataFrame(
    population_rows,
    columns=["grid_id", "population"]
)

print("Rows:", len(population_df))
print(
    "Missing:",
    population_df["population"].isna().sum()
)

print("\nPopulation summary:")
print(population_df["population"].describe())

print(
    "\nZero-population cells:",
    (population_df["population"] == 0).sum()
)

In [ ]:
# ----------------------------------------
# Dominant FBFM40 class in each 5 km cell
# ----------------------------------------

fbfm_path = "Missouri_FBFM40_2024.tif"

fbfm_rows = []

with rasterio.open(fbfm_path) as src:

    print("FBFM40 CRS:", src.crs)
    print("Resolution:", src.res)

    for _, row in grid.iterrows():

        # Extract FBFM40 pixels inside the grid cell
        out_image, _ = mask(
            src,
            [row.geometry],
            crop=True,
            filled=False
        )

        # Get valid fuel-model pixels
        values = out_image[0].compressed()

        # Remove LANDFIRE NoData value
        values = values[values != src.nodata]

        if len(values) > 0:

            # Find dominant fuel-model class
            classes, counts = np.unique(
                values.astype(int),
                return_counts=True
            )

            dominant_fbfm = classes[
                np.argmax(counts)
            ]

        else:
            dominant_fbfm = np.nan

        fbfm_rows.append([
            row["grid_id"],
            dominant_fbfm
        ])

# Create FBFM40 table
fbfm_df = pd.DataFrame(
    fbfm_rows,
    columns=["grid_id", "fbfm40"]
)

print("\nRows:", len(fbfm_df))
print("Missing:", fbfm_df["fbfm40"].isna().sum())

print("\nDominant FBFM40 classes:")
print(
    fbfm_df["fbfm40"]
    .value_counts()
    .sort_index()
)

## 3. Recorded-wildfire response and final preprocessing

The original master CSV was saved in original cell 90 **before** five continuous
predictor rows were filled and the single NLCD class-0 cell was corrected. That
historical file is preserved and must not be mistaken for the cleaned modeling
matrix. The optional checkpoint after correction below uses a new derived path.
Category codes are converted to strings before the correction; one-hot encoding
then occurs once. No scaling or additional predictor transformation is introduced.

In [ ]:
# ----------------------------------------
# Build master 5 km modeling dataset
# ----------------------------------------

# Start with grid ID and wildfire response
model_df = grid[
    ["grid_id", "fire_count"]
].copy()

# Add continuous environmental predictors
model_df = model_df.merge(
    continuous_df,
    on="grid_id",
    how="left"
)

# Add dominant NLCD class
model_df = model_df.merge(
    landcover_df,
    on="grid_id",
    how="left"
)

# Add population
model_df = model_df.merge(
    population_df,
    on="grid_id",
    how="left"
)

# Add dominant FBFM40 class
model_df = model_df.merge(
    fbfm_df,
    on="grid_id",
    how="left"
)

print("Dataset shape:", model_df.shape)
print("\nColumns:")
print(model_df.columns.tolist())

print("\nMissing values:")
print(model_df.isna().sum())

In [ ]:
# ----------------------------------------
# Create binary wildfire response
# ----------------------------------------

# 1 = at least one recorded wildfire
# 0 = no recorded wildfire
model_df["fire_presence"] = (
    model_df["fire_count"] > 0
).astype(int)

print("\nFire presence distribution:")
print(model_df["fire_presence"].value_counts())

print("\nFire presence percentage:")
print(
    model_df["fire_presence"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
# ----------------------------------------
# Fill 5 missing cells from nearest valid cell
# ----------------------------------------

from scipy.spatial import cKDTree
import numpy as np

continuous_cols = [
    "elevation",
    "slope",
    "northness",
    "eastness",
    "canopy",
    "ndvi",
    "temperature",
    "precipitation",
    "pdsi",
    "wind",
    "road_distance"
]

# Identify valid and missing rows
valid_mask = model_df[continuous_cols].notna().all(axis=1)
missing_mask = ~valid_mask

# Get grid geometry for each model row
geometry_lookup = grid.set_index("grid_id").geometry

# Calculate centroid coordinates
coords = np.array([
    [
        geometry_lookup.loc[gid].centroid.x,
        geometry_lookup.loc[gid].centroid.y
    ]
    for gid in model_df["grid_id"]
])

# Build nearest-neighbor tree using valid cells
tree = cKDTree(coords[valid_mask])

# Find nearest valid cell for each missing cell
distances, indices = tree.query(coords[missing_mask], k=1)

# Get corresponding model_df indices
valid_indices = model_df.index[valid_mask]
missing_indices = model_df.index[missing_mask]

# Fill predictors from nearest valid cell
for missing_idx, neighbor_position in zip(
    missing_indices,
    indices
):
    neighbor_idx = valid_indices[neighbor_position]

    model_df.loc[
        missing_idx,
        continuous_cols
    ] = model_df.loc[
        neighbor_idx,
        continuous_cols
    ].values

# Report distances in kilometers
print("Filled cells:", len(missing_indices))
print(
    "Nearest-neighbor distances (km):",
    np.round(distances / 1000, 2)
)

print("\nRemaining missing values:")
print(model_df[continuous_cols].isna().sum())

In [ ]:
# ----------------------------------------
# Summary of continuous predictors
# ----------------------------------------

continuous_cols = [
    "elevation",
    "slope",
    "northness",
    "eastness",
    "canopy",
    "ndvi",
    "temperature",
    "precipitation",
    "pdsi",
    "wind",
    "road_distance",
    "population"
]

summary = model_df[continuous_cols].describe().T

# Keep the most useful statistics
print(
    summary[
        ["min", "mean", "std", "max"]
    ].round(3)
)

In [ ]:
# ----------------------------------------
# Check predictor correlations
# ----------------------------------------

correlation = model_df[continuous_cols].corr()

# Find highly correlated predictor pairs
high_corr = []

for i in range(len(continuous_cols)):
    for j in range(i + 1, len(continuous_cols)):

        value = correlation.iloc[i, j]

        if abs(value) >= 0.80:
            high_corr.append([
                continuous_cols[i],
                continuous_cols[j],
                round(value, 3)
            ])

print("Highly correlated pairs (|r| >= 0.80):")

if high_corr:
    for pair in high_corr:
        print(pair)
else:
    print("None")

### Category-code preparation

The initial, superseded one-hot encoding from original cell 94 was consolidated
into the final encoding after the class-0 correction (original cell 97). The old
37-column output, which included `lc_0`, remains in the archived original; it is
not a result of the conversion-only cell below.

In [ ]:
# ----------------------------------------
# Convert categorical codes before correcting NLCD class 0
# ----------------------------------------

# Convert numeric codes to categorical labels
model_df["landcover"] = (
    model_df["landcover"]
    .astype(int)
    .astype(str)
)

model_df["fbfm40"] = (
    model_df["fbfm40"]
    .astype(int)
    .astype(str)
)



In [ ]:
# ----------------------------------------
# Inspect the NLCD class-0 cell
# ----------------------------------------

lc0 = model_df[
    model_df["landcover"] == "0"
][["grid_id", "fire_count", "landcover", "fbfm40"]]

print(lc0)

In [ ]:
# ----------------------------------------
# Fix the single invalid NLCD class-0 cell
# ----------------------------------------

from scipy.spatial import cKDTree
import numpy as np

# Cell with invalid NLCD class 0
invalid_idx = model_df.index[
    model_df["landcover"] == "0"
][0]

invalid_grid_id = model_df.loc[invalid_idx, "grid_id"]

# Get centroid of the invalid cell
invalid_geom = grid.loc[
    grid["grid_id"] == invalid_grid_id,
    "geometry"
].iloc[0]

invalid_xy = np.array([
    invalid_geom.centroid.x,
    invalid_geom.centroid.y
])

# Get cells with valid NLCD classes
valid_lc = model_df[
    model_df["landcover"] != "0"
].copy()

valid_geoms = grid.set_index("grid_id").loc[
    valid_lc["grid_id"],
    "geometry"
]

# Get centroid coordinates of valid cells
valid_xy = np.array([
    [geom.centroid.x, geom.centroid.y]
    for geom in valid_geoms
])

# Find nearest valid cell
tree = cKDTree(valid_xy)

distance, position = tree.query(
    invalid_xy,
    k=1
)

# Get its land-cover class
neighbor_idx = valid_lc.index[position]
neighbor_class = model_df.loc[
    neighbor_idx,
    "landcover"
]

# Replace class 0
model_df.loc[
    invalid_idx,
    "landcover"
] = neighbor_class

print("Grid ID:", invalid_grid_id)
print("Nearest-cell distance (km):", round(distance / 1000, 2))
print("Assigned NLCD class:", neighbor_class)

### Optional cleaned, unencoded checkpoint

This new export is disabled by default and has not been executed. It preserves
the historical pre-correction CSV and writes the corrected `model_df` to a new
path only when explicitly enabled. No predictor values are changed here.

In [ ]:
SAVE_CLEANED_CHECKPOINT = False

if SAVE_CLEANED_CHECKPOINT:
    cleaned_path = Path("data/derived/missouri_wildfire_model_cleaned.csv")
    cleaned_path.parent.mkdir(parents=True, exist_ok=True)
    model_df.to_csv(cleaned_path, index=False)
    print("Cleaned, unencoded modeling checkpoint saved:", cleaned_path)

In [ ]:
# ----------------------------------------
# One-hot encode the corrected modeling dataset
# ----------------------------------------

model_encoded = pd.get_dummies(
    model_df,
    columns=["landcover", "fbfm40"],
    prefix=["lc", "fuel"],
    dtype=int
)

print("Encoded shape:", model_encoded.shape)

print(
    "lc_0 exists:",
    "lc_0" in model_encoded.columns
)

In [ ]:
# ----------------------------------------
# Create 50 km spatial validation blocks
# ----------------------------------------

# Get centroid coordinates of each 5 km grid cell
grid_centroids = grid.copy()

grid_centroids["x"] = grid_centroids.geometry.centroid.x
grid_centroids["y"] = grid_centroids.geometry.centroid.y

# 50 km = 50,000 meters
block_size = 50000

# Assign each cell to a 50 km x 50 km spatial block
grid_centroids["block_x"] = (
    grid_centroids["x"] // block_size
).astype(int)

grid_centroids["block_y"] = (
    grid_centroids["y"] // block_size
).astype(int)

# Create unique block ID
grid_centroids["block_id"] = (
    grid_centroids["block_x"].astype(str)
    + "_"
    + grid_centroids["block_y"].astype(str)
)

# Add block ID to modeling data
model_encoded = model_encoded.merge(
    grid_centroids[["grid_id", "block_id"]],
    on="grid_id",
    how="left"
)

print(
    "Number of spatial blocks:",
    model_encoded["block_id"].nunique()
)

print("\nCells per block:")
print(
    model_encoded["block_id"]
    .value_counts()
    .describe()
)

In [ ]:
# ----------------------------------------
# Check fire distribution across blocks
# ----------------------------------------

block_summary = (
    model_encoded
    .groupby("block_id")
    .agg(
        cells=("grid_id", "size"),
        fire_cells=("fire_presence", "sum"),
        fires=("fire_count", "sum")
    )
)

# Percentage of cells with fire occurrence
block_summary["fire_presence_pct"] = (
    block_summary["fire_cells"]
    / block_summary["cells"]
    * 100
)

print(block_summary.describe().round(2))

print(
    "\nBlocks with no fire-positive cells:",
    (block_summary["fire_cells"] == 0).sum()
)

print(
    "Blocks with no zero-fire cells:",
    (
        block_summary["fire_cells"]
        == block_summary["cells"]
    ).sum()
)

## 4. Original predictors and baseline validation comparisons

Use the explicit, ordered 33 original predictor columns from `scripts/project.py`.
This preserves the original feature matrix and excludes identifiers, target,
spatial groups, and every saved prediction column when loading a checkpoint.
The first spatial baseline uses `GroupKFold`; the subsequent stratified spatial
analysis intentionally uses a different splitter. Do not substitute one for the
other when comparing the reported AUC values.

In [ ]:
# ----------------------------------------
# Prepare Random Forest modeling data
# ----------------------------------------

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd
import numpy as np

# Binary response
y = model_encoded["fire_presence"]

# Spatial groups
groups = model_encoded["block_id"]

# Explicit original predictors exclude saved predictions and identifiers.
from scripts.project import FEATURES

X = model_encoded.loc[:, FEATURES].copy()
assert X.shape == (7229, 33), "Expected the original 7,229 cells and 33 predictors."
assert X.columns.tolist() == FEATURES, "Original predictor order must be preserved."
assert not X.isna().any().any(), "Original fitted predictors must contain no missing values."

print("Samples:", X.shape[0])
print("Predictors:", X.shape[1])
print("Positive cells:", y.sum())
print("Negative cells:", (y == 0).sum())

In [ ]:
# ----------------------------------------
# Check predictor columns
# ----------------------------------------

print("Number of predictors:", len(X.columns))

for i, col in enumerate(X.columns, start=1):
    print(i, col)

In [ ]:
# ----------------------------------------
# Baseline spatial 5-fold CV: GroupKFold (historical mean AUC 0.673)
# ----------------------------------------

spatial_cv = GroupKFold(n_splits=5)

results = []

for fold, (train_idx, test_idx) in enumerate(
    spatial_cv.split(X, y, groups),
    start=1
):

    # Split by spatial blocks
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    # Baseline Random Forest
    rf = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )

    # Train model
    rf.fit(X_train, y_train)

    # Predictions
    y_pred = rf.predict(X_test)
    y_prob = rf.predict_proba(X_test)[:, 1]

    # Calculate evaluation metrics
    results.append({
        "fold": fold,
        "roc_auc": roc_auc_score(y_test, y_prob),
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    })

# Convert results to table
cv_results = pd.DataFrame(results)

print(cv_results.round(3))

print("\nMean performance:")
print(
    cv_results[
        ["roc_auc", "accuracy", "precision", "recall", "f1"]
    ]
    .mean()
    .round(3)
)

print("\nStandard deviation:")
print(
    cv_results[
        ["roc_auc", "accuracy", "precision", "recall", "f1"]
    ]
    .std()
    .round(3)
)

In [ ]:
# ----------------------------------------
# Baseline stratified random 5-fold CV versus GroupKFold spatial CV
# ----------------------------------------

from sklearn.model_selection import StratifiedKFold

random_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

random_results = []

for fold, (train_idx, test_idx) in enumerate(
    random_cv.split(X, y),
    start=1
):

    # Random train/test split
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    # Same baseline model for fair comparison
    rf = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    # Predictions
    y_pred = rf.predict(X_test)
    y_prob = rf.predict_proba(X_test)[:, 1]

    random_results.append({
        "fold": fold,
        "roc_auc": roc_auc_score(y_test, y_prob),
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    })

random_cv_results = pd.DataFrame(random_results)

print(random_cv_results.round(3))

print("\nRandom CV mean:")
print(
    random_cv_results[
        ["roc_auc", "accuracy", "precision", "recall", "f1"]
    ].mean().round(3)
)

print("\nSpatial CV mean:")
print(
    cv_results[
        ["roc_auc", "accuracy", "precision", "recall", "f1"]
    ].mean().round(3)
)

In [ ]:
# ----------------------------------------
# Separate stratified spatial baseline: historical mean AUC 0.657
# ----------------------------------------

from sklearn.model_selection import StratifiedGroupKFold

spatial_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

stratified_results = []

for fold, (train_idx, test_idx) in enumerate(
    spatial_cv.split(X, y, groups),
    start=1
):

    # Keep entire spatial blocks together
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    # Baseline Random Forest
    rf = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    # Predictions
    y_pred = rf.predict(X_test)
    y_prob = rf.predict_proba(X_test)[:, 1]

    # Save fold performance
    stratified_results.append({
        "fold": fold,
        "test_cells": len(test_idx),
        "positive_rate": y_test.mean(),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    })

stratified_cv_results = pd.DataFrame(
    stratified_results
)

print(stratified_cv_results.round(3))

print("\nMean performance:")
print(
    stratified_cv_results[
        ["roc_auc", "accuracy", "precision", "recall", "f1"]
    ].mean().round(3)
)

print("\nStandard deviation:")
print(
    stratified_cv_results[
        ["roc_auc", "accuracy", "precision", "recall", "f1"]
    ].std().round(3)
)

## 5. Global spatial tuning and nonnested model comparisons

These grid searches use all rows with five stratified spatial folds. The following
OOF evaluations reuse parameters selected on those same folds, so they are
**nonnested** evaluations. The selected RF configuration supplies `best_rf`, the
original SHAP analysis, and the full-data mapping model. These analyses and saved
results are retained; nested validation appears separately below.

In [ ]:
# ----------------------------------------
# Select final RF configuration with global spatial grid search (nonnested)
# ----------------------------------------

from sklearn.model_selection import GridSearchCV

# Focused RF parameter grid
param_grid = {
    "n_estimators": [300, 500],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 3, 5],
    "max_features": ["sqrt", 0.5],
    "class_weight": [None, "balanced"]
}

# Random Forest model
rf_model = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

# Spatially stratified grid search
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=spatial_cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

# Keep spatial blocks together during CV
grid_search.fit(
    X,
    y,
    groups=groups
)

print("Best parameters:")
print(grid_search.best_params_)

print(
    "\nBest spatial CV ROC-AUC:",
    round(grid_search.best_score_, 3)
)

In [ ]:
# ----------------------------------------
# Nonnested RF evaluation using globally selected parameters
# ----------------------------------------

from sklearn.base import clone

# Best RF from the spatial grid search
best_rf = grid_search.best_estimator_

# Store predictions for every grid cell
oof_prob = np.zeros(len(X))
oof_pred = np.zeros(len(X), dtype=int)

fold_results = []

for fold, (train_idx, test_idx) in enumerate(
    spatial_cv.split(X, y, groups),
    start=1
):

    # Create a fresh copy of the tuned model
    model = clone(best_rf)

    # Train only on the training spatial blocks
    model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    # Predict held-out spatial blocks
    prob = model.predict_proba(
        X.iloc[test_idx]
    )[:, 1]

    pred = model.predict(
        X.iloc[test_idx]
    )

    # Save out-of-fold predictions
    oof_prob[test_idx] = prob
    oof_pred[test_idx] = pred

    # Calculate fold metrics
    fold_results.append({
        "fold": fold,
        "roc_auc": roc_auc_score(
            y.iloc[test_idx], prob
        ),
        "accuracy": accuracy_score(
            y.iloc[test_idx], pred
        ),
        "precision": precision_score(
            y.iloc[test_idx], pred
        ),
        "recall": recall_score(
            y.iloc[test_idx], pred
        ),
        "f1": f1_score(
            y.iloc[test_idx], pred
        )
    })

# Results by fold
tuned_cv_results = pd.DataFrame(fold_results)

print(tuned_cv_results.round(3))

print("\nMean performance:")
print(
    tuned_cv_results[
        ["roc_auc", "accuracy", "precision", "recall", "f1"]
    ].mean().round(3)
)

print("\nStandard deviation:")
print(
    tuned_cv_results[
        ["roc_auc", "accuracy", "precision", "recall", "f1"]
    ].std().round(3)
)

# Overall out-of-fold AUC
print(
    "\nOverall out-of-fold ROC-AUC:",
    round(roc_auc_score(y, oof_prob), 3)
)

In [ ]:
# ----------------------------------------
# Save RF out-of-fold predictions
# ----------------------------------------

# Add nonnested RF OOF predictions from globally selected parameters
model_encoded["rf_oof_prob"] = oof_prob
model_encoded["rf_oof_pred"] = oof_pred

# Check the new columns
print(
    model_encoded[
        ["grid_id", "fire_presence",
         "rf_oof_prob", "rf_oof_pred"]
    ].head()
)

# Save checkpoint
model_encoded.to_csv(
    "data/missouri_wildfire_model_encoded.csv",
    index=False
)

print("\nSaved model table.")

In [ ]:
# ----------------------------------------
# Baseline XGBoost with spatial CV
# ----------------------------------------

from xgboost import XGBClassifier
from sklearn.base import clone

# Baseline XGBoost classifier
xgb_base = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

# Store out-of-fold predictions
xgb_oof_prob = np.zeros(len(X))
xgb_oof_pred = np.zeros(len(X), dtype=int)

xgb_fold_results = []

# Use the stratified spatial splitter from original cell 104
for fold, (train_idx, test_idx) in enumerate(
    spatial_cv.split(X, y, groups),
    start=1
):

    # Fresh model for each fold
    model = clone(xgb_base)

    # Train on spatial training blocks
    model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    # Predict held-out spatial blocks
    prob = model.predict_proba(
        X.iloc[test_idx]
    )[:, 1]

    pred = (prob >= 0.5).astype(int)

    # Save out-of-fold predictions
    xgb_oof_prob[test_idx] = prob
    xgb_oof_pred[test_idx] = pred

    # Calculate fold metrics
    xgb_fold_results.append({
        "fold": fold,
        "roc_auc": roc_auc_score(
            y.iloc[test_idx], prob
        ),
        "accuracy": accuracy_score(
            y.iloc[test_idx], pred
        ),
        "precision": precision_score(
            y.iloc[test_idx], pred
        ),
        "recall": recall_score(
            y.iloc[test_idx], pred
        ),
        "f1": f1_score(
            y.iloc[test_idx], pred
        )
    })

# Convert results to table
xgb_cv_results = pd.DataFrame(xgb_fold_results)

print(xgb_cv_results.round(3))

print("\nMean performance:")
print(
    xgb_cv_results[
        ["roc_auc", "accuracy", "precision", "recall", "f1"]
    ].mean().round(3)
)

print("\nStandard deviation:")
print(
    xgb_cv_results[
        ["roc_auc", "accuracy", "precision", "recall", "f1"]
    ].std().round(3)
)

print(
    "\nOverall out-of-fold ROC-AUC:",
    round(roc_auc_score(y, xgb_oof_prob), 3)
)

In [ ]:
# ----------------------------------------
# Global spatial XGBoost grid search (nonnested)
# ----------------------------------------

from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

# Base XGBoost model
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

# Focused parameter grid
xgb_param_grid = {
    "n_estimators": [300, 500],
    "max_depth": [3, 5],
    "learning_rate": [0.03, 0.05],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5]
}

# Spatial grid search
xgb_grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=xgb_param_grid,
    scoring="roc_auc",
    cv=spatial_cv,
    n_jobs=-1,
    verbose=1
)

# Keep spatial blocks together
xgb_grid_search.fit(
    X,
    y,
    groups=groups
)

print("Best parameters:")
print(xgb_grid_search.best_params_)

print(
    "\nBest spatial CV ROC-AUC:",
    round(xgb_grid_search.best_score_, 3)
)

In [ ]:
# ----------------------------------------
# Nonnested XGBoost evaluation using globally selected parameters
# ----------------------------------------

from sklearn.base import clone

# Best XGBoost from spatial tuning
best_xgb = xgb_grid_search.best_estimator_

# Store out-of-fold predictions
xgb_tuned_oof_prob = np.zeros(len(X))
xgb_tuned_oof_pred = np.zeros(len(X), dtype=int)

fold_results = []

# Use exactly the same spatial folds
for fold, (train_idx, test_idx) in enumerate(
    spatial_cv.split(X, y, groups),
    start=1
):

    # Fresh copy of tuned model
    model = clone(best_xgb)

    # Train on spatial training blocks
    model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    # Predict held-out spatial blocks
    prob = model.predict_proba(
        X.iloc[test_idx]
    )[:, 1]

    pred = (prob >= 0.5).astype(int)

    # Save OOF predictions
    xgb_tuned_oof_prob[test_idx] = prob
    xgb_tuned_oof_pred[test_idx] = pred

    # Calculate fold metrics
    fold_results.append({
        "fold": fold,
        "roc_auc": roc_auc_score(
            y.iloc[test_idx], prob
        ),
        "accuracy": accuracy_score(
            y.iloc[test_idx], pred
        ),
        "precision": precision_score(
            y.iloc[test_idx], pred
        ),
        "recall": recall_score(
            y.iloc[test_idx], pred
        ),
        "f1": f1_score(
            y.iloc[test_idx], pred
        )
    })

# Results table
xgb_tuned_results = pd.DataFrame(fold_results)

print(xgb_tuned_results.round(3))

print("\nMean performance:")
print(
    xgb_tuned_results[
        ["roc_auc", "accuracy", "precision", "recall", "f1"]
    ].mean().round(3)
)

print("\nStandard deviation:")
print(
    xgb_tuned_results[
        ["roc_auc", "accuracy", "precision", "recall", "f1"]
    ].std().round(3)
)

print(
    "\nOverall out-of-fold ROC-AUC:",
    round(roc_auc_score(y, xgb_tuned_oof_prob), 3)
)

In [ ]:
# ----------------------------------------
# Save tuned XGBoost predictions
# ----------------------------------------

# Add nonnested XGBoost OOF predictions from globally selected parameters
model_encoded["xgb_oof_prob"] = xgb_tuned_oof_prob
model_encoded["xgb_oof_pred"] = xgb_tuned_oof_pred

# Save updated checkpoint
model_encoded.to_csv(
    "data/missouri_wildfire_model_encoded.csv",
    index=False
)

print(
    model_encoded[
        [
            "grid_id",
            "fire_presence",
            "rf_oof_prob",
            "xgb_oof_prob"
        ]
    ].head()
)

print("\nUpdated model table saved.")

## 6. Original full-data RF interpretation

SHAP uses `shap.TreeExplainer(best_rf)` and interprets `shap_values[:, :, 1]`,
explicitly the recorded-wildfire class. The original constructor does not specify
feature perturbation, a background dataset, or `model_output`; none is inferred
or changed here. The displayed exploratory plots are historical. Publication
labels are applied separately without changing model features or SHAP values.

In [ ]:
# ----------------------------------------
# Random Forest feature importance
# ----------------------------------------

# Get built-in RF importance
rf_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": best_rf.feature_importances_
})

# Rank from most to least important
rf_importance = rf_importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

# Show top 20 predictors
print(rf_importance.head(20).round(4))

In [ ]:
# ----------------------------------------
# SHAP analysis for tuned Random Forest
# ----------------------------------------

import shap

# Create Tree SHAP explainer
explainer = shap.TreeExplainer(best_rf)

# Calculate SHAP values for all grid cells
shap_values = explainer.shap_values(X)

# Check output structure
print("SHAP output type:", type(shap_values))
print("SHAP output shape:", np.array(shap_values).shape)

In [ ]:
# ----------------------------------------
# SHAP importance for wildfire class
# ----------------------------------------

# Select SHAP values for class 1 (fire presence)
shap_fire = shap_values[:, :, 1]

# Mean absolute SHAP value for each predictor
mean_abs_shap = np.abs(shap_fire).mean(axis=0)

# Create importance table
shap_importance = pd.DataFrame({
    "feature": X.columns,
    "mean_abs_shap": mean_abs_shap
})

# Rank predictors
shap_importance = shap_importance.sort_values(
    "mean_abs_shap",
    ascending=False
).reset_index(drop=True)

# Show top 20
print(shap_importance.head(20).round(4))

In [ ]:
# ----------------------------------------
# SHAP summary plot for wildfire class
# ----------------------------------------

import matplotlib.pyplot as plt

# Show direction and magnitude of SHAP effects
shap.summary_plot(
    shap_fire,
    X,
    max_display=20,
    show=False
)

plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------
# SHAP dependence plots
# ----------------------------------------

# Key continuous predictors to inspect
features_to_plot = [
    "population",
    "ndvi",
    "canopy",
    "precipitation",
    "road_distance",
    "temperature"
]

for feature in features_to_plot:

    # Plot predictor value against SHAP effect
    shap.dependence_plot(
        feature,
        shap_fire,
        X,
        interaction_index=None,
        show=False
    )

    plt.title(f"SHAP dependence: {feature}")
    plt.tight_layout()
    plt.show()

## 7. Full-data model, relative susceptibility, and original exports

`final_rf = clone(best_rf)` retains the earlier spatial-grid-search selection and
is fitted to all 7,229 cells. This is distinct from the nested performance
estimation. Class-1 model scores are divided into the original five equal-frequency
classes. The following in-sample and nonnested OOF summaries retain their original
calculations; they are not substituted for nested OOF validation.

In [ ]:
# ----------------------------------------
# Final Random Forest susceptibility model
# ----------------------------------------

from sklearn.base import clone

# Retain parameters selected by the original global spatial grid search
final_rf = clone(best_rf)

# Train using all available grid cells
final_rf.fit(X, y)

# Class-1 model score for recorded wildfire presence; used as relative susceptibility
rf_susceptibility = final_rf.predict_proba(X)[:, 1]

# Check the relative susceptibility score distribution
print(
    pd.Series(rf_susceptibility)
    .describe()
    .round(3)
)

In [ ]:
# ----------------------------------------
# Attach susceptibility to 5-km grid
# ----------------------------------------

# Create prediction table
prediction_df = pd.DataFrame({
    "grid_id": model_encoded["grid_id"],
    "rf_susceptibility": rf_susceptibility,
    "rf_oof_prob": model_encoded["rf_oof_prob"]
})

# Join predictions to geographic grid
susceptibility_grid = grid.merge(
    prediction_df,
    on="grid_id",
    how="left"
)

print(
    susceptibility_grid[
        ["grid_id", "fire_count",
         "rf_susceptibility", "rf_oof_prob"]
    ].head()
)

print(
    "\nMissing susceptibility values:",
    susceptibility_grid["rf_susceptibility"].isna().sum()
)

In [ ]:
# ----------------------------------------
# Continuous wildfire susceptibility map
# ----------------------------------------

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 9))

# Plot continuous RF susceptibility
susceptibility_grid.plot(
    column="rf_susceptibility",
    cmap="YlOrRd",
    linewidth=0,
    legend=True,
    vmin=0,
    vmax=1,
    ax=ax,
    legend_kwds={
        "label": "Model score for recorded wildfire presence",
        "shrink": 0.7
    }
)

# Add Missouri state boundary
missouri_5070.boundary.plot(
    ax=ax,
    color="black",
    linewidth=0.8
)

ax.set_title(
    "Relative Wildfire Susceptibility\nMissouri, 2021–2025",
    fontsize=14
)

ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------
# Create wildfire susceptibility classes
# ----------------------------------------

# Divide predictions into five equal-frequency classes
susceptibility_grid["susceptibility_class"] = pd.qcut(
    susceptibility_grid["rf_susceptibility"],
    q=5,
    labels=[
        "Very Low",
        "Low",
        "Moderate",
        "High",
        "Very High"
    ]
)

# Show relative susceptibility score thresholds
thresholds = susceptibility_grid[
    "rf_susceptibility"
].quantile([0, 0.2, 0.4, 0.6, 0.8, 1.0])

print("Susceptibility thresholds:")
print(thresholds.round(3))

print("\nCells per class:")
print(
    susceptibility_grid[
        "susceptibility_class"
    ].value_counts(sort=False)
)

In [ ]:
# ----------------------------------------
# In-sample recorded wildfire activity by full-model susceptibility class
# ----------------------------------------

class_validation = (
    susceptibility_grid
    .groupby(
        "susceptibility_class",
        observed=False
    )
    .agg(
        cells=("grid_id", "count"),
        fire_cells=("fire_count", lambda x: (x > 0).sum()),
        total_fires=("fire_count", "sum"),
        mean_fires=("fire_count", "mean")
    )
)

# Percentage of cells containing >= 1 fire
class_validation["fire_cell_pct"] = (
    class_validation["fire_cells"]
    / class_validation["cells"]
    * 100
)

# Percentage of all fires in each class
class_validation["fire_share_pct"] = (
    class_validation["total_fires"]
    / class_validation["total_fires"].sum()
    * 100
)

print(class_validation.round(2))

In [ ]:
# ----------------------------------------
# Nonnested tuned RF OOF susceptibility classes
# ----------------------------------------

# Retain original quintiles of nonnested OOF scores
susceptibility_grid["oof_class"] = pd.qcut(
    susceptibility_grid["rf_oof_prob"],
    q=5,
    labels=[
        "Very Low",
        "Low",
        "Moderate",
        "High",
        "Very High"
    ]
)

# Summarize observed fires by OOF class
oof_validation = (
    susceptibility_grid
    .groupby("oof_class", observed=False)
    .agg(
        cells=("grid_id", "count"),
        fire_cells=("fire_count", lambda x: (x > 0).sum()),
        total_fires=("fire_count", "sum"),
        mean_fires=("fire_count", "mean")
    )
)

# Percentage of cells containing fire
oof_validation["fire_cell_pct"] = (
    oof_validation["fire_cells"]
    / oof_validation["cells"]
    * 100
)

# Percentage of all observed fires
oof_validation["fire_share_pct"] = (
    oof_validation["total_fires"]
    / oof_validation["total_fires"].sum()
    * 100
)

# Show OOF thresholds
print("OOF susceptibility thresholds:")
print(
    susceptibility_grid["rf_oof_prob"]
    .quantile([0, .2, .4, .6, .8, 1])
    .round(3)
)

print("\nOOF validation:")
print(oof_validation.round(2))

In [ ]:
# ----------------------------------------
# Save OOF validation table
# ----------------------------------------

oof_validation.to_csv(
    "data/rf_oof_susceptibility_validation.csv"
)

print("OOF validation table saved.")

In [ ]:
# ----------------------------------------
# Plot OOF validation by susceptibility
# ----------------------------------------

import matplotlib.pyplot as plt

# Preserve susceptibility class order
class_order = [
    "Very Low",
    "Low",
    "Moderate",
    "High",
    "Very High"
]

plot_data = oof_validation.loc[class_order]

fig, ax = plt.subplots(figsize=(8, 5))

# Plot percentage of fire-positive cells
ax.bar(
    class_order,
    plot_data["fire_cell_pct"]
)

ax.set_xlabel("Nonnested OOF Relative Susceptibility Class")
ax.set_ylabel("Cells with Recorded Wildfire (%)")
ax.set_title(
    "Recorded Wildfire Presence by Relative Susceptibility Class"
)

ax.set_ylim(0, 65)

# Add percentage labels
for i, value in enumerate(plot_data["fire_cell_pct"]):
    ax.text(
        i,
        value + 1,
        f"{value:.1f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------
# Export final wildfire susceptibility grid
# ----------------------------------------

# Keep important mapping and validation fields
export_grid = susceptibility_grid[
    [
        "grid_id",
        "fire_count",
        "rf_susceptibility",
        "rf_oof_prob",
        "susceptibility_class",
        "oof_class",
        "geometry"
    ]
].copy()

# Shorter field names are convenient in GIS
export_grid = export_grid.rename(
    columns={
        "rf_susceptibility": "rf_susc",
        "rf_oof_prob": "rf_oof"
    }
)

# Convert categorical fields to strings
export_grid["susceptibility_class"] = (
    export_grid["susceptibility_class"].astype(str)
)

export_grid["oof_class"] = (
    export_grid["oof_class"].astype(str)
)

# Export as GeoPackage
export_grid.to_file(
    "data/Missouri_Wildfire_Susceptibility_5km.gpkg",
    layer="wildfire_susceptibility",
    driver="GPKG"
)

print("GeoPackage saved.")

print("\nCRS:", export_grid.crs)
print("Number of cells:", len(export_grid))
print("Missing RF susceptibility:",
      export_grid["rf_susc"].isna().sum())

print("\nFields:")
print(export_grid.columns.tolist())

In [ ]:
# ----------------------------------------
# Save final models and key results
# ----------------------------------------

import os
import joblib

# Create output folders if needed
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

# Save final tuned Random Forest
joblib.dump(
    final_rf,
    "models/random_forest_wildfire_5km.joblib"
)

# Fit final tuned XGBoost on all cells
final_xgb = clone(best_xgb)
final_xgb.fit(X, y)

# Save final tuned XGBoost
joblib.dump(
    final_xgb,
    "models/xgboost_wildfire_5km.joblib"
)

# Save RF SHAP importance
shap_importance.to_csv(
    "results/rf_shap_importance.csv",
    index=False
)

# Save nonnested tuned RF spatial CV results
tuned_cv_results.to_csv(
    "results/rf_spatial_cv_results.csv",
    index=False
)

# Save nonnested tuned XGBoost spatial CV results
xgb_tuned_results.to_csv(
    "results/xgb_spatial_cv_results.csv",
    index=False
)

# Save OOF susceptibility validation
oof_validation.to_csv(
    "results/rf_oof_susceptibility_validation.csv"
)

print("Checkpoint complete.")
print("RF model saved.")
print("XGBoost model saved.")
print("SHAP importance saved.")
print("Spatial CV results saved.")
print("OOF validation saved.")

## 8. Exploratory wildfire-count analysis

The original XGBoost count model, mean-count comparator, and rank summaries are
retained as an explicitly exploratory analysis. They do not replace the binary
recorded-wildfire models or their validation metrics.

In [ ]:
# ----------------------------------------
# Inspect wildfire count distribution
# ----------------------------------------

fire_count = model_df["fire_count"]

print("Fire-count summary:")
print(fire_count.describe().round(2))

print("\nZero-fire cells:",
      (fire_count == 0).sum())

print("Zero-fire percentage:",
      round((fire_count == 0).mean() * 100, 2))

print("\nVariance:",
      round(fire_count.var(), 2))

print("Mean:",
      round(fire_count.mean(), 2))

print("Variance / Mean:",
      round(fire_count.var() / fire_count.mean(), 2))

# Selected percentiles
print("\nPercentiles:")
print(
    fire_count.quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99]
    ).round(2)
)

In [ ]:
# ----------------------------------------
# Baseline XGBoost fire-count model
# ----------------------------------------

from xgboost import XGBRegressor
from sklearn.base import clone
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

# Fire-count response
y_count = model_encoded["fire_count"]

# Baseline count model
xgb_count = XGBRegressor(
    objective="count:poisson",
    n_estimators=500,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# Store spatial OOF predictions
count_oof_pred = np.zeros(len(X))

count_results = []

# Use the same spatial folds
for fold, (train_idx, test_idx) in enumerate(
    spatial_cv.split(X, y, groups),
    start=1
):

    # Fresh model for each fold
    model = clone(xgb_count)

    # Train on fire counts
    model.fit(
        X.iloc[train_idx],
        y_count.iloc[train_idx]
    )

    # Predict held-out spatial blocks
    pred = model.predict(X.iloc[test_idx])

    # Count predictions cannot be negative
    pred = np.maximum(pred, 0)

    # Save OOF predictions
    count_oof_pred[test_idx] = pred

    # Calculate fold metrics
    count_results.append({
        "fold": fold,
        "mae": mean_absolute_error(
            y_count.iloc[test_idx], pred
        ),
        "rmse": np.sqrt(
            mean_squared_error(
                y_count.iloc[test_idx], pred
            )
        )
    })

# Results table
count_cv_results = pd.DataFrame(count_results)

print(count_cv_results.round(3))

print("\nMean performance:")
print(
    count_cv_results[
        ["mae", "rmse"]
    ].mean().round(3)
)

print("\nOverall OOF performance:")
print(
    "MAE:",
    round(mean_absolute_error(y_count, count_oof_pred), 3)
)

print(
    "RMSE:",
    round(
        np.sqrt(
            mean_squared_error(y_count, count_oof_pred)
        ),
        3
    )
)

In [ ]:
# ----------------------------------------
# Compare XGBoost with mean-count baseline
# ----------------------------------------

baseline_oof_pred = np.zeros(len(X))
baseline_results = []

for fold, (train_idx, test_idx) in enumerate(
    spatial_cv.split(X, y, groups),
    start=1
):

    # Mean count from training cells only
    train_mean = y_count.iloc[train_idx].mean()

    # Predict same training mean for test cells
    pred = np.full(len(test_idx), train_mean)

    baseline_oof_pred[test_idx] = pred

    # Calculate baseline metrics
    baseline_results.append({
        "fold": fold,
        "mae": mean_absolute_error(
            y_count.iloc[test_idx], pred
        ),
        "rmse": np.sqrt(
            mean_squared_error(
                y_count.iloc[test_idx], pred
            )
        )
    })

baseline_results = pd.DataFrame(baseline_results)

print("Mean-count baseline:")
print(baseline_results.round(3))

print("\nMean baseline performance:")
print(
    baseline_results[
        ["mae", "rmse"]
    ].mean().round(3)
)

# Compare overall OOF performance
baseline_mae = mean_absolute_error(
    y_count, baseline_oof_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_count, baseline_oof_pred
    )
)

xgb_mae = mean_absolute_error(
    y_count, count_oof_pred
)

xgb_rmse = np.sqrt(
    mean_squared_error(
        y_count, count_oof_pred
    )
)

print("\nOverall OOF comparison:")
print(f"Baseline MAE: {baseline_mae:.3f}")
print(f"XGBoost MAE:  {xgb_mae:.3f}")

print(f"\nBaseline RMSE: {baseline_rmse:.3f}")
print(f"XGBoost RMSE:  {xgb_rmse:.3f}")

print("\nImprovement over baseline:")
print(
    "MAE:",
    round((baseline_mae - xgb_mae)
          / baseline_mae * 100, 1),
    "%"
)

print(
    "RMSE:",
    round((baseline_rmse - xgb_rmse)
          / baseline_rmse * 100, 1),
    "%"
)

In [ ]:
# ----------------------------------------
# Evaluate ranking of fire-count predictions
# ----------------------------------------

from scipy.stats import spearmanr

# Overall spatial OOF rank correlation
rho, p_value = spearmanr(
    y_count,
    count_oof_pred
)

print("XGBoost spatial OOF:")
print(f"Spearman rho: {rho:.3f}")
print(f"P-value: {p_value:.3e}")

# Calculate correlation separately by fold
rank_results = []

for fold, (_, test_idx) in enumerate(
    spatial_cv.split(X, y, groups),
    start=1
):

    rho_fold, _ = spearmanr(
        y_count.iloc[test_idx],
        count_oof_pred[test_idx]
    )

    rank_results.append({
        "fold": fold,
        "spearman_rho": rho_fold
    })

rank_results = pd.DataFrame(rank_results)

print("\nBy spatial fold:")
print(rank_results.round(3))

print("\nMean fold Spearman rho:")
print(
    round(rank_results["spearman_rho"].mean(), 3)
)

In [ ]:
# ----------------------------------------
# Save exploratory count-model results
# ----------------------------------------

count_cv_results["spearman_rho"] = (
    rank_results["spearman_rho"]
)

count_cv_results.to_csv(
    "results/xgb_count_spatial_cv_results.csv",
    index=False
)

count_summary = pd.DataFrame({
    "metric": [
        "Baseline OOF MAE",
        "XGBoost OOF MAE",
        "MAE improvement (%)",
        "Baseline OOF RMSE",
        "XGBoost OOF RMSE",
        "RMSE improvement (%)",
        "OOF Spearman rho"
    ],
    "value": [
        baseline_mae,
        xgb_mae,
        (baseline_mae - xgb_mae) / baseline_mae * 100,
        baseline_rmse,
        xgb_rmse,
        (baseline_rmse - xgb_rmse) / baseline_rmse * 100,
        rho
    ]
})

count_summary.to_csv(
    "results/xgb_count_summary.csv",
    index=False
)

print(count_summary.round(3))
print("\nCount-model experiment saved.")

## 9. Nested RF spatial validation

The original RF outer/inner splitters, seeds, hyperparameter grid, fitting, and
threshold behavior are retained. Only predictor selection is made explicit to
prevent saved prediction columns from entering a resumed analysis. Original code
for the combined nested RF/XGBoost files was absent from the archived notebook;
new selective reproduction is documented separately in `scripts/reproduce_validation.py`.

In [ ]:
# ----------------------------------------
# Set up nested spatial cross-validation
# ----------------------------------------

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold

# Explicit original predictors also exclude nested OOF columns in saved checkpoints.
from scripts.project import FEATURES

X_nested = model_encoded.loc[:, FEATURES].copy()
assert X_nested.shape == (7229, 33), "Expected the original modeling matrix."
assert X_nested.columns.tolist() == FEATURES, "Original predictor order must be preserved."
assert not X_nested.isna().any().any(), "Original fitted predictors must contain no missing values."
assert X_nested.equals(X), "Nested and baseline analyses must use the same original predictors."

# Binary wildfire-presence target
y_nested = model_encoded["fire_presence"]

# 50-km spatial groups
groups_nested = model_encoded["block_id"]

# Outer CV evaluates the complete RF tuning procedure
outer_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Nested CV setup complete.")
print("Cells:", len(X_nested))
print("Predictors:", X_nested.shape[1])
print("Spatial blocks:", groups_nested.nunique())

print("\nTarget distribution:")
print(
    y_nested.value_counts(normalize=True)
    .sort_index()
    .round(3)
)

# Check outer-fold composition
outer_summary = []

for fold, (train_idx, test_idx) in enumerate(
    outer_cv.split(
        X_nested,
        y_nested,
        groups_nested
    ),
    start=1
):

    train_groups = set(
        groups_nested.iloc[train_idx]
    )

    test_groups = set(
        groups_nested.iloc[test_idx]
    )

    outer_summary.append({
        "fold": fold,
        "train_cells": len(train_idx),
        "test_cells": len(test_idx),
        "test_positive_rate":
            y_nested.iloc[test_idx].mean(),
        "train_blocks": len(train_groups),
        "test_blocks": len(test_groups),
        "overlap_blocks":
            len(train_groups & test_groups)
    })

outer_summary = pd.DataFrame(outer_summary)

print("\nOuter-fold structure:")
print(outer_summary.round(3))

In [ ]:
# ----------------------------------------
# Nested spatial CV - Random Forest
# ----------------------------------------

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Focused RF parameter grid
nested_param_grid = {
    "n_estimators": [300, 500],
    "max_depth": [8, 10, None],
    "min_samples_leaf": [1, 3],
    "max_features": ["sqrt"],
    "class_weight": ["balanced"]
}

# Store held-out spatial outer-fold predictions
nested_oof_prob = np.zeros(len(X_nested))
nested_oof_pred = np.zeros(len(X_nested), dtype=int)

nested_results = []
nested_best_params = []

# Outer spatial folds
for fold, (train_idx, test_idx) in enumerate(
    outer_cv.split(
        X_nested,
        y_nested,
        groups_nested
    ),
    start=1
):

    print(f"\n===== Outer Fold {fold} =====")

    X_train = X_nested.iloc[train_idx]
    X_test = X_nested.iloc[test_idx]

    y_train = y_nested.iloc[train_idx]
    y_test = y_nested.iloc[test_idx]

    groups_train = groups_nested.iloc[train_idx]

    # Inner spatial CV for tuning
    inner_cv = StratifiedGroupKFold(
        n_splits=4,
        shuffle=True,
        random_state=42 + fold
    )

    # Base RF
    rf = RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    )

    # Tune only inside outer training data
    search = GridSearchCV(
        estimator=rf,
        param_grid=nested_param_grid,
        scoring="roc_auc",
        cv=inner_cv,
        n_jobs=-1,
        refit=True
    )

    search.fit(
        X_train,
        y_train,
        groups=groups_train
    )

    # Best inner-tuned model
    model = search.best_estimator_

    # Predict untouched outer test fold
    prob = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)

    nested_oof_prob[test_idx] = prob
    nested_oof_pred[test_idx] = pred

    # Save outer-fold metrics
    nested_results.append({
        "fold": fold,
        "roc_auc": roc_auc_score(y_test, prob),
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred)
    })

    # Save selected parameters
    nested_best_params.append({
        "fold": fold,
        "inner_best_auc": search.best_score_,
        **search.best_params_
    })

    print(
        "Inner best AUC:",
        round(search.best_score_, 3)
    )

    print(
        "Outer AUC:",
        round(roc_auc_score(y_test, prob), 3)
    )

# Convert results to tables
nested_results = pd.DataFrame(nested_results)
nested_best_params = pd.DataFrame(nested_best_params)

print("\n===== Outer-fold performance =====")
print(nested_results.round(3))

print("\nMean:")
print(
    nested_results[
        ["roc_auc", "accuracy",
         "precision", "recall", "f1"]
    ].mean().round(3)
)

print("\nStd:")
print(
    nested_results[
        ["roc_auc", "accuracy",
         "precision", "recall", "f1"]
    ].std().round(3)
)

print("\nOverall nested OOF AUC:")
print(
    round(
        roc_auc_score(
            y_nested,
            nested_oof_prob
        ),
        3
    )
)

print("\nSelected parameters:")
print(nested_best_params)

In [ ]:
# ----------------------------------------
# Save nested spatial CV results
# ----------------------------------------

# Add held-out spatial nested OOF predictions
model_encoded["rf_nested_oof_prob"] = nested_oof_prob
model_encoded["rf_nested_oof_pred"] = nested_oof_pred

# Save fold-level performance
nested_results.to_csv(
    "results/rf_nested_spatial_cv_results.csv",
    index=False
)

# Save parameters selected within each outer fold
nested_best_params.to_csv(
    "results/rf_nested_best_params.csv",
    index=False
)

# Save updated modeling table
model_encoded.to_csv(
    "data/missouri_wildfire_model_encoded.csv",
    index=False
)

print("Nested spatial CV results saved.")
print("Overall nested OOF AUC:",
      round(roc_auc_score(y_nested, nested_oof_prob), 3))
print("Missing nested predictions:",
      np.isnan(nested_oof_prob).sum())

wrap up

## 10. Historical manuscript tables

These cells preserve the original table generators and cached outputs. Table 1
contains the original source claims, not independently verified product versions.
Original Table 3 contains manually entered rounded performance values. Revised
publication artifacts use documented saved inputs outside these legacy generators.

In [ ]:
# ----------------------------------------
# Table 1 - Predictor datasets
# ----------------------------------------

predictor_table = pd.DataFrame([
    ["Elevation", "USGS 3DEP", "Static",
     "10 m", "250 m", "Mean"],

    ["Slope", "Derived from USGS 3DEP", "Static",
     "10 m", "250 m", "Mean"],

    ["Northness", "Derived from aspect", "Static",
     "10 m", "250 m", "Mean"],

    ["Eastness", "Derived from aspect", "Static",
     "10 m", "250 m", "Mean"],

    ["Tree canopy cover", "GTAC Tree Canopy Cover",
     "2021", "Fine resolution", "250 m", "Mean"],

    ["NDVI", "MODIS MOD13Q1",
     "2021-2025 (Apr-Oct)", "250 m",
     "250 m", "Seasonal/multi-year mean"],

    ["Land cover", "NLCD",
     "2021", "30 m", "250 m",
     "Dominant class"],

    ["Temperature", "PRISM",
     "2021-2025", "~4 km", "250 m export",
     "Multi-year mean"],

    ["Precipitation", "PRISM",
     "2021-2025", "~4 km", "250 m export",
     "Mean annual precipitation"],

    ["PDSI", "GRIDMET Drought",
     "2021-2025", "~4 km", "250 m export",
     "Multi-year mean"],

    ["Wind speed", "GRIDMET",
     "2021-2025", "~4 km", "250 m export",
     "Multi-year mean"],

    ["Population", "WorldPop",
     "2020", "100 m", "100 m",
     "Sum"],

    ["Distance to roads", "TIGER/Line Roads",
     "2016", "Vector", "250 m raster",
     "Mean distance"],

    ["Fuel model", "LANDFIRE FBFM40",
     "2024", "30 m", "30 m",
     "Dominant class"]
], columns=[
    "Predictor",
    "Source",
    "Temporal_Coverage",
    "Native_Resolution",
    "Processing_Resolution",
    "Aggregation_to_5km"
])

# Save reproducibility table
predictor_table.to_csv(
    "results/table1_predictor_datasets.csv",
    index=False
)

print(predictor_table.to_string(index=False))
print("\nTable 1 saved.")

In [ ]:
# ----------------------------------------
# Table 2 - Nested spatial CV performance
# ----------------------------------------

metrics = [
    "roc_auc",
    "accuracy",
    "precision",
    "recall",
    "f1"
]

metric_names = {
    "roc_auc": "ROC-AUC",
    "accuracy": "Accuracy",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1-score"
}

# Summarize outer-fold performance
table2 = pd.DataFrame({
    "Metric": [metric_names[m] for m in metrics],
    "Mean": [nested_results[m].mean() for m in metrics],
    "SD": [nested_results[m].std() for m in metrics],
    "Minimum": [nested_results[m].min() for m in metrics],
    "Maximum": [nested_results[m].max() for m in metrics]
})

# Round for manuscript presentation
table2.iloc[:, 1:] = table2.iloc[:, 1:].round(3)

# Save table
table2.to_csv(
    "results/table2_nested_spatial_cv.csv",
    index=False
)

print(table2.to_string(index=False))

print(
    "\nOverall nested OOF ROC-AUC:",
    round(
        roc_auc_score(
            y_nested,
            nested_oof_prob
        ),
        3
    )
)

print("\nTable 2 saved.")

In [ ]:
# ----------------------------------------
# Table 3 - Model and validation comparison
# ----------------------------------------

table3 = pd.DataFrame([
    {
        "Analysis": "Random vs spatial validation",
        "Model": "Random Forest",
        "Validation": "Random 5-fold CV",
        "ROC_AUC": 0.768,
        "Accuracy": 0.707,
        "Precision": 0.679,
        "Recall": 0.559,
        "F1": 0.612
    },
    {
        "Analysis": "Random vs spatial validation",
        "Model": "Random Forest",
        "Validation": "Spatial 5-fold CV",
        "ROC_AUC": 0.673,
        "Accuracy": 0.641,
        "Precision": 0.584,
        "Recall": 0.488,
        "F1": 0.525
    },
    {
        "Analysis": "Tuned model comparison",
        "Model": "Random Forest",
        "Validation": "Spatial 5-fold CV",
        "ROC_AUC": 0.663,
        "Accuracy": 0.625,
        "Precision": 0.544,
        "Recall": 0.607,
        "F1": 0.573
    },
    {
        "Analysis": "Tuned model comparison",
        "Model": "XGBoost",
        "Validation": "Spatial 5-fold CV",
        "ROC_AUC": 0.649,
        "Accuracy": 0.614,
        "Precision": 0.546,
        "Recall": 0.427,
        "F1": 0.474
    }
])

# Save manuscript table
table3.to_csv(
    "results/table3_model_validation_comparison.csv",
    index=False
)

print(table3.to_string(index=False))
print("\nTable 3 saved.")

In [ ]:
# ----------------------------------------
# Table 4 - Nested OOF susceptibility validation
# ----------------------------------------

# Create quintile classes from nested OOF predictions
class_labels = [
    "Very Low",
    "Low",
    "Moderate",
    "High",
    "Very High"
]

model_encoded["nested_oof_class"] = pd.qcut(
    model_encoded["rf_nested_oof_prob"],
    q=5,
    labels=class_labels
)

# Summarize observed wildfire occurrence
table4 = (
    model_encoded
    .groupby(
        "nested_oof_class",
        observed=True
    )
    .agg(
        Cells=("grid_id", "count"),
        Fire_Positive_Cells=("fire_presence", "sum"),
        Total_Fires=("fire_count", "sum"),
        Mean_Fires_Per_Cell=("fire_count", "mean")
    )
    .reset_index()
)

# Percentage of cells with at least one fire
table4["Fire_Positive_Cells_Pct"] = (
    table4["Fire_Positive_Cells"]
    / table4["Cells"] * 100
)

# Percentage of all observed fires
table4["Fire_Share_Pct"] = (
    table4["Total_Fires"]
    / table4["Total_Fires"].sum() * 100
)

# Round presentation fields
table4["Mean_Fires_Per_Cell"] = (
    table4["Mean_Fires_Per_Cell"].round(2)
)

table4["Fire_Positive_Cells_Pct"] = (
    table4["Fire_Positive_Cells_Pct"].round(2)
)

table4["Fire_Share_Pct"] = (
    table4["Fire_Share_Pct"].round(2)
)

# Save final validation table
table4.to_csv(
    "results/table4_nested_oof_susceptibility_validation.csv",
    index=False
)

print(table4.to_string(index=False))

# Show quintile thresholds
print("\nNested OOF susceptibility thresholds:")
print(
    model_encoded["rf_nested_oof_prob"]
    .quantile([0, .2, .4, .6, .8, 1])
    .round(3)
)

print("\nTable 4 saved.")

## 11. Historical figure generators and original numbering

The following original figure cells and cached images are retained as provenance.
Their historical numbering and filenames are not the current manuscript numbering:
original Figure 3A–C supplies the SHAP analysis underlying revised Figure 4;
original Figure 4 is the susceptibility map. The revised `Figure4_SHAP_summary`
files present at cleanup had no corresponding generator in the original notebook.
The old validation chart contains hardcoded rounded means without error bars.
Use the separate publication script for final files, rather than rerunning these
legacy cells or reconstructing a beeswarm from mean SHAP importance alone.

In [ ]:
# ----------------------------------------
# Historical Figure 2 - Rounded mean model validation comparison
# ----------------------------------------

import matplotlib.pyplot as plt
import numpy as np

# Performance values
labels = [
    "RF\nRandom CV",
    "RF\nSpatial CV",
    "RF Tuned\nSpatial CV",
    "XGBoost Tuned\nSpatial CV",
    "RF Nested\nSpatial CV"
]

auc = [
    0.768,
    0.673,
    0.663,
    0.649,
    0.662
]

x = np.arange(len(labels))

# Create figure
fig, ax = plt.subplots(figsize=(9, 5.5))

bars = ax.bar(
    x,
    auc,
    width=0.65
)

# Add values above bars
for bar, value in zip(bars, auc):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.008,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=10
    )

# Format axes
ax.set_ylabel("ROC-AUC", fontsize=11)
ax.set_xlabel("")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)

ax.set_ylim(0.50, 0.82)

ax.set_title(
    "Wildfire Susceptibility Model Performance",
    fontsize=13
)

ax.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

# Save publication-quality figure
plt.savefig(
    "results/figure2_model_validation.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ----------------------------------------
# Historical Figure 3A - Original SHAP beeswarm
# ----------------------------------------

import matplotlib.pyplot as plt
import shap

plt.figure(figsize=(9, 7))

shap.summary_plot(
    shap_fire,
    X,
    max_display=15,
    show=False
)

plt.title(
    "Random Forest SHAP Summary",
    fontsize=13
)

plt.tight_layout()

plt.savefig(
    "results/figure3a_shap_beeswarm.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ----------------------------------------
# Historical Figure 3B - Original NDVI SHAP dependence
# ----------------------------------------

plt.figure(figsize=(7, 5))

shap.dependence_plot(
    "ndvi",
    shap_fire,
    X,
    interaction_index=None,
    show=False
)

plt.title(
    "SHAP Dependence: NDVI",
    fontsize=12
)

plt.tight_layout()

plt.savefig(
    "results/figure3b_shap_ndvi.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ----------------------------------------
# Historical Figure 3C - Original canopy SHAP dependence
# ----------------------------------------

plt.figure(figsize=(7, 5))

shap.dependence_plot(
    "canopy",
    shap_fire,
    X,
    interaction_index=None,
    show=False
)

plt.title(
    "SHAP Dependence: Tree Canopy Cover",
    fontsize=12
)

plt.tight_layout()

plt.savefig(
    "results/figure3c_shap_canopy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ----------------------------------------
# Historical Figure 4 - Full-model relative susceptibility map
# ----------------------------------------

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Class order
class_order = [
    "Very Low",
    "Low",
    "Moderate",
    "High",
    "Very High"
]

# Convert class labels to ordered category
susceptibility_grid["susceptibility_class"] = pd.Categorical(
    susceptibility_grid["susceptibility_class"],
    categories=class_order,
    ordered=True
)

# Convert categories to numeric codes for plotting
susceptibility_grid["class_code"] = (
    susceptibility_grid["susceptibility_class"].cat.codes
)

# Five-class susceptibility palette
cmap = ListedColormap([
    "#2E7D32",
    "#8BC34A",
    "#FDD835",
    "#FB8C00",
    "#D32F2F"
])

# Create figure
fig, ax = plt.subplots(figsize=(8, 9))

susceptibility_grid.plot(
    column="class_code",
    cmap=cmap,
    linewidth=0,
    ax=ax
)

# Missouri outline
missouri_5070.boundary.plot(
    ax=ax,
    linewidth=0.8
)

ax.set_title(
    "Relative Wildfire Susceptibility in Missouri",
    fontsize=14
)

ax.set_axis_off()

# Create manual legend
legend_handles = [
    plt.Line2D(
        [0], [0],
        marker="s",
        linestyle="",
        markersize=10,
        markerfacecolor=cmap(i),
        markeredgecolor="none",
        label=label
    )
    for i, label in enumerate(class_order)
]

ax.legend(
    handles=legend_handles,
    title="Relative Susceptibility",
    loc="lower left",
    frameon=True
)

plt.tight_layout()

# Save publication-quality image
plt.savefig(
    "results/figure4_missouri_wildfire_susceptibility.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ----------------------------------------
# Historical Figure 5 - Nested OOF class validation
# ----------------------------------------

import matplotlib.pyplot as plt
import numpy as np

# Keep class order
class_order = [
    "Very Low",
    "Low",
    "Moderate",
    "High",
    "Very High"
]

plot_df = (
    table4
    .set_index("nested_oof_class")
    .loc[class_order]
    .reset_index()
)

x = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(8, 5.5))

bars = ax.bar(
    x,
    plot_df["Fire_Positive_Cells_Pct"],
    width=0.65
)

# Add percentages above bars
for bar, value in zip(
    bars,
    plot_df["Fire_Positive_Cells_Pct"]
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.1f}%",
        ha="center",
        va="bottom",
        fontsize=10
    )

ax.set_xticks(x)
ax.set_xticklabels(class_order)

ax.set_ylabel(
    "Grid Cells with Recorded Wildfire (%)"
)

ax.set_xlabel(
    "Nested OOF Susceptibility Class"
)

ax.set_ylim(
    0,
    plot_df["Fire_Positive_Cells_Pct"].max() + 10
)

ax.set_title(
    "Recorded Wildfire Presence by Relative Susceptibility Class"
)

ax.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    "results/figure5_nested_oof_validation.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# HISTORICAL FIGURE 1: Study area, recorded wildfire, and spatial CV
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold

# ------------------------------------------------------------
# 1. Prepare grid with response and spatial-block information
# ------------------------------------------------------------

# Make a plotting copy
fig_grid = grid.copy()

# Add response/block information if not already in grid
if "fire_presence" not in fig_grid.columns:
    fig_grid = fig_grid.merge(
        model_encoded[["grid_id", "fire_presence", "block_id"]],
        on="grid_id",
        how="left"
    )

# ------------------------------------------------------------
# 2. Reconstruct the 5 outer spatial folds
# ------------------------------------------------------------

# Reconstruct the stratified outer splitter, not the GroupKFold baseline
outer_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

X_dummy = np.zeros((len(fig_grid), 1))
y_plot = fig_grid["fire_presence"].values
groups_plot = fig_grid["block_id"].values

# Assign each test set its outer-fold number
fig_grid["cv_fold"] = np.nan

for fold, (_, test_idx) in enumerate(
    outer_cv.split(X_dummy, y_plot, groups_plot),
    start=1
):
    fig_grid.iloc[test_idx,
                  fig_grid.columns.get_loc("cv_fold")] = fold

fig_grid["cv_fold"] = fig_grid["cv_fold"].astype(int)

# ------------------------------------------------------------
# 3. Create a separate layer for the original 5-km squares
# ------------------------------------------------------------

# grid_square preserves the original equal 5-km cells
grid_squares = fig_grid.copy()

if "grid_square" in grid_squares.columns:
    grid_squares = grid_squares.set_geometry("grid_square")

# ------------------------------------------------------------
# 4. Prepare wildfire points
# ------------------------------------------------------------

# Make sure wildfire data use the same CRS
fires_plot = fires_model.to_crs(fig_grid.crs)

# ------------------------------------------------------------
# 5. Create the two-panel publication figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 7),
    constrained_layout=True
)

# ============================================================
# PANEL A: Study area and wildfire occurrence
# ============================================================

ax = axes[0]

# Light 5-km modeling grid
grid_squares.boundary.plot(
    ax=ax,
    linewidth=0.15,
    alpha=0.25
)

# Missouri boundary
missouri_5070.boundary.plot(
    ax=ax,
    linewidth=1.2
)

# Wildfire occurrence points
fires_plot.plot(
    ax=ax,
    markersize=2,
    alpha=0.35
)

ax.set_title(
    "(a) Wildfire occurrence and 5-km modeling grid",
    fontsize=12
)

ax.set_axis_off()

# ============================================================
# PANEL B: Spatial cross-validation folds
# ============================================================

ax = axes[1]

# Plot cells by outer spatial fold
fig_grid.plot(
    column="cv_fold",
    categorical=True,
    legend=True,
    ax=ax,
    linewidth=0.05,
    edgecolor="white",
    legend_kwds={
        "title": "Outer CV Fold",
        "loc": "lower left"
    }
)

# Missouri outline
missouri_5070.boundary.plot(
    ax=ax,
    linewidth=1.2
)

ax.set_title(
    "(b) Spatial cross-validation folds",
    fontsize=12
)

ax.set_axis_off()

# ------------------------------------------------------------
# 6. Save publication-quality version
# ------------------------------------------------------------

plt.savefig(
    "results/figure1_study_area_spatial_cv.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 12. Separate publication and selective reproduction entry points

Publication figures are generated from preserved or explicitly reproduced
artifacts by `scripts/make_publication_figures.py`. Descriptive SHAP labels affect
plotting only; they do not rename, reorder, or transform the predictor matrix.
The original notebook saved mean absolute SHAP importance but no per-cell SHAP
array, so a beeswarm requires the original array or a documented recomputation
from the preserved fitted RF. No model retuning is implied by figure generation.

Selective validation and fold persistence are handled by
`scripts/reproduce_validation.py`. Its explicit run options keep baseline random
CV, baseline GroupKFold spatial CV, and nested spatial validation separate.

After activating the documented project environment, the artifact-based publication
command below generates the six final figures without fitting a model. SHAP
preparation is a separate explicit command that uses the saved RF and does not
retune or refit it:

```bash
python scripts/make_publication_figures.py --figures 1 2 3 4 5 6
# Only when the saved per-cell SHAP artifact needs preparation:
python scripts/make_publication_figures.py --prepare-shap
```

Selective validation is also explicit. Artifact mode does not fit models;
baseline and nested-XGBoost modes do. None is automatically invoked here:

```bash
python scripts/reproduce_validation.py --mode artifacts
# Explicitly requested refits only:
python scripts/reproduce_validation.py --mode baselines --workers 4
python scripts/reproduce_validation.py --mode nested-xgb --workers 4
python scripts/reproduce_validation.py --help
```

These are manual commands, not notebook execution cells. See the publication and
reproduction documentation for the completed artifacts and actual execution log.